In [0]:
from pyspark.sql.functions import *
import datetime
from delta.tables import DeltaTable

## Variables

In [0]:
adls_account = dbutils.widgets.get("adls_account_name")
adls_container = dbutils.widgets.get("adls_container")
mongo_conn = dbutils.secrets.get("mongo_connection_string")

## Cargamos las compras desde MongoDB

In [0]:
mongo_uri = dbutils.secrets.get(scope='akv-kvfinalprojecteastus001', key='conn-mongodb-atlas')

### Obtenemos los watermarks para la carga incremental

In [0]:
if spark.catalog.tableExists("finalproject.bronze.watermarks"):
  maximo_presencial = spark.sql('select max_fecha from finalproject.bronze.watermarks where collection = "presencial"').first()['max_fecha']
  maximo_online = spark.sql('select max_fecha from finalproject.bronze.watermarks where collection = "online"').first()['max_fecha']
else:
  # Creamos la tabla watermarks porque no existe
  spark.sql("""
        CREATE TABLE IF NOT EXISTS finalproject.bronze.watermarks (
            collection STRING,
            max_fecha   TIMESTAMP
        )
    """)
  # Insertar los valores iniciales
  spark.sql(f"""
        INSERT INTO finalproject.bronze.watermarks
        VALUES ('presencial', '1900-01-01'), ('online', '1900-01-01')
    """)
  maximo_presencial = "1900-01-01"
  maximo_online = "1900-01-01"

### Extraemos la colección online de MongoDB

In [0]:
pipeline = f"""[
    {{ "$match": {{ "created_at": {{ "$gt": {{ "$date": "{maximo_presencial}" }} }} }} }}
]"""

# Leemos usando la opción "aggregation.pipeline"
df_presencial = spark.read \
    .format("mongodb") \
    .option("spark.mongodb.read.connection.uri", mongo_uri) \
    .option("spark.mongodb.read.database", "maximo_db") \
    .option("spark.mongodb.read.collection", "presencial") \
    .option("aggregation.pipeline", pipeline) \
    .load()

### Extraemos la colección presencial de MongoDB

In [0]:
pipeline = f"""[
    {{ "$match": {{ "updated_at": {{ "$gt": {{ "$date": "{maximo_online}" }} }} }} }}
]"""

# Leemos usando la opción "aggregation.pipeline"
df_online = spark.read \
    .format("mongodb") \
    .option("spark.mongodb.read.connection.uri", mongo_uri) \
    .option("spark.mongodb.read.database", "maximo_db") \
    .option("spark.mongodb.read.collection", "online") \
    .option("aggregation.pipeline", pipeline) \
    .load()

In [0]:
df_online.createTempView('df_online_sql')
df_presencial.createTempView('df_presencial_sql')

### Hay un error en las colecciones, la colección online contiene tanto las compras online, como presencial, para quedarnos con las que sí son online, debemos hacer un antijoin

In [0]:
df_online = df_online.join(df_presencial, on="ventaid", how="left_anti")

### Unimos las compras presencial y online

In [0]:
df_compras = df_online.unionByName(df_presencial, allowMissingColumns=True)

### Realizamos la carga incremental

In [0]:
if not spark.catalog.tableExists('finalproject.bronze.compras'):
  compras_historicos = spark.table('finalproject.bronze.compras_historicos')
  compras = compras_historicos.unionByName(df_compras, allowMissingColumns=True)
  compras.write.format("delta").mode('overwrite').saveAsTable('finalproject.bronze.compras')
else:
    compras = DeltaTable.forName(spark,'finalproject.bronze.compras')
    compras.alias('hist')\
    .merge(
        df_compras.alias('inc'),
        'hist._id = inc._id'
    ).whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()

## Cargamos los detalles desde ADLS

In [0]:
# Solo corre si necesitas reiniciar el Autoloader
#dbutils.fs.rm("/Volumes/finalproject/bronze/detalles/checkpoint/autoloader/1/", recurse=True)
#dbutils.fs.rm("/Volumes/finalproject/bronze/detalles/checkpoint/autoloader_w/", recurse=True)

In [0]:
def hacer_merge_incremental(micro_batch_df, batch_id):
    micro_batch_df = micro_batch_df.toDF(*[col.lower() for col in micro_batch_df.columns])
    
    
    if not spark.catalog.tableExists('finalproject.bronze.detalles'):
        detalles_historico = spark.table('finalproject.bronze.detalles_historicos')
        detalles = detalles_historico.unionByName(micro_batch_df, allowMissingColumns=True)
        detalles = detalles.withColumn('fecha_carga', current_timestamp())
        detalles.write.format("delta").mode('overwrite').saveAsTable('finalproject.bronze.detalles')
    else:
        detalles = DeltaTable.forName(spark, 'finalproject.bronze.detalles')
        micro_batch_df = micro_batch_df.withColumn('fecha_carga', current_timestamp())
        # IDs que ya existen en la tabla (irán por whenMatchedUpdateAll)
        ids_existentes = detalles.toDF().select("detalle_id")
        matched = micro_batch_df.join(ids_existentes, on="detalle_id", how="inner")
        not_matched = micro_batch_df.join(ids_existentes, on="detalle_id", how="left_anti")
        
        print(f"[Batch {batch_id}] Total en micro_batch: {micro_batch_df.count()}")
        print(f"[Batch {batch_id}] Irán por whenMatchedUpdateAll (ya existen): {matched.count()}")
        print(f"[Batch {batch_id}] Irán por whenNotMatchedInsertAll (nuevos): {not_matched.count()}")

        detalles.alias('hist') \
            .merge(
                micro_batch_df.alias('inc'),
                'hist.detalle_id = inc.detalle_id'
            ).whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()

In [0]:
df_detalles = (
    spark
    .readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("pathGlobFilter", "*.csv")
    .option("header", "true")
    .option("sep",";")
    .option("cloudFiles.schemaLocation", "/Volumes/finalproject/bronze/detalles/checkpoint/autoloader/1/")
    .load(f"abfss://{adls_container}@{adls_account}.dfs.core.windows.net/actual/")
)

In [0]:
# Tomamos esa misma variable (df_detalles) y le decimos dónde "desembocar" el agua
flujo_escritura = (
    df_detalles.writeStream
    .trigger(availableNow=True)
    .foreachBatch(hacer_merge_incremental) # Aquí llamamos a tu función del IF/ELSE
    .option("checkpointLocation", "/Volumes/finalproject/bronze/detalles/checkpoint/autoloader_w/")
    .start()
)